The code is based on: https://github.com/parthsarthi03/raptor

# Terminal commands

In [15]:
!git clone https://github.com/parthsarthi03/raptor

Cloning into 'raptor'...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 89 (delta 43), reused 24 (delta 24), pack-reused 26 (from 1)
Receiving objects: 100% (89/89), 850.72 KiB | 13.95 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [5]:
!pip install gdown

# Imports + Constants

In [1]:
with open('/home/oh/arubinstein17/.config/hugging_face/hf.yaml', 'r') as file:
    token = file.read()
    token = token.split(":")[1].strip()

In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
import gdown
import torch
from transformers import AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
import torch
from huggingface_hub import login
import numpy as np
import random
import logging
import pickle
import tiktoken
from typing import (
    Any,
    Callable,
    Dict,
    List,
    Optional,
    Set,
    Tuple
)
import copy


PATH_TO_RAPTOR = "./raptor"
HF_TOKEN = token # FILL IN
RANDOM_SEED = 42
TREE_PATH = "tree.pkl"
SAMPLE_TEXT_PATH = os.path.join(os.path.abspath(''), 'sample.txt')
GRANULARITY = 100
SUMMARY_LENGTH = 200
if not os.path.exists(SAMPLE_TEXT_PATH):
    gdown.download(
        f"https://drive.google.com/uc?export=download&confirm=pbef&id=1_sE53mgV_RptRv8LXl4IPrAeruTfmw3-",
        SAMPLE_TEXT_PATH
    )

# import from raptor repo
sys.path.insert(
    0,
    PATH_TO_RAPTOR
)
from raptor import (
    BaseSummarizationModel,
    BaseQAModel,
    BaseEmbeddingModel,
    RetrievalAugmentationConfig,
)
from raptor.Retrievers import (
    BaseRetriever
)
from raptor.utils import (
    reverse_mapping,
    get_node_list,
    distances_from_embeddings,
    indices_of_nearest_neighbors_from_distances,
    get_text,
    split_text
)
from raptor.cluster_tree_builder import (
    ClusterTreeConfig
)
sys.path.pop(0);

# Infrastructure

In [4]:
# You can define your own Summarization model by extending the base Summarization Class.
class GEMMASummarizationModel(BaseSummarizationModel):
    def __init__(self, model_name="google/gemma-2b-it"):
        # Initialize the tokenizer and the pipeline for the GEMMA model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.summarization_pipeline = pipeline(
            "text-generation",
            model=model_name,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),  # Use "cpu" if CUDA is not available
        )

    def summarize(self, context, max_tokens=150):
        # Format the prompt for summarization
        messages=[
            {"role": "user", "content": f"Write a summary of the following, including as many key details as possible: {context}:"}
        ]

        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Generate the summary using the pipeline
        apply_random_seed(RANDOM_SEED)
        outputs = self.summarization_pipeline(
            prompt,
            max_new_tokens=max_tokens,
            do_sample=False,
            temperature=0.7,
            top_k=50,
            top_p=0.95
        )

        # Extracting and returning the generated summary
        summary = outputs[0]["generated_text"].strip()
        # remove technical prefix
        split = summary.split("start_of_turn>model\n")
        summarization_prompt = "\n\n".join(split[:-1])
        summary = split[-1].strip()
        return summary, summarization_prompt


class GEMMAQAModel(BaseQAModel):
    def __init__(self, model_name= "google/gemma-2b-it"):
        # Initialize the tokenizer and the pipeline for the model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            "text-generation",
            model=model_name,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        )

    def answer_question(self, context, question):
        # Apply the chat template for the context and question
        messages=[
              {"role": "user", "content": f"Given Context: {context} Give the best full answer amongst the option to question {question}"}
        ]
        print("Context: ", context)
        print("Question: ", question)
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Generate the answer using the pipeline
        outputs = self.qa_pipeline(
            prompt,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.7,
            top_k=50,
            top_p=0.95
        )

        # Extracting and returning the generated answer
        answer = outputs[0]["generated_text"][len(prompt):]
        return answer


class SBertEmbeddingModel(BaseEmbeddingModel):
    def __init__(self, model_name="sentence-transformers/multi-qa-mpnet-base-cos-v1"):
        self.model = SentenceTransformer(model_name)

    def create_embedding(self, text):
        return self.model.encode(text)

    def __call__(self, text):
        return self.create_embedding(text)


def apply_random_seed(random_seed):
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # to suppress warning
    torch.use_deterministic_algorithms(True, warn_only=True)


class RetrievalAugmentation:
    """
    A Retrieval Augmentation class that combines the TreeBuilder and TreeRetriever classes.
    Enables adding documents to the tree, retrieving information, and answering questions.
    """

    def __init__(
        self,
        config=None,
        tree=None,
        embedding_models=None,
        cluster_embedding_model=None,
        retrieve_context_func=None,
        make_database_builder_func=None
    ):
        """
        Initializes a RetrievalAugmentation instance with the specified configuration.
        Args:
            config (RetrievalAugmentationConfig): The configuration for the RetrievalAugmentation instance.
            tree: The tree instance or the path to a pickled tree file.
            retrieve_context_func: The function to retrieve the context for the question.
            ??
        """
        if config is None:
            config = RetrievalAugmentationConfig(
                tb_embedding_models=embedding_models,
                tb_cluster_embedding_model=cluster_embedding_model
            )
        if not isinstance(config, RetrievalAugmentationConfig):
            raise ValueError(
                "config must be an instance of RetrievalAugmentationConfig"
            )

        if isinstance(tree, str) and os.path.exists(tree):
            try:
                with open(tree, "rb") as file:
                    self.tree = pickle.load(file)

            except Exception as e:
                raise ValueError(f"Failed to load tree from {tree}: {e}")
        elif isinstance(tree, str) and not os.path.exists(tree):
            self.tree = None
        else:
            self.tree = tree

        assert retrieve_context_func is not None, "retrieve_context_func must be provided"
        assert make_database_builder_func is not None, "make_database_builder_func must be provided"

        self.tree_builder = make_database_builder_func()

        self.tree_retriever_config = config.tree_retriever_config
        self.qa_model = config.qa_model

        self.retrieve_context_func = retrieve_context_func

        if self.tree is not None:
            self.retriever = TreeRetriever(
                self.tree_retriever_config,
                self.tree,
                retrieve_context_func=self.retrieve_context_func
            )
        else:
            self.retriever = None

        logging.info(
            f"Successfully initialized RetrievalAugmentation with Config {config.log_config()}"
        )

    def add_documents(self, docs):
        """
        Adds documents to the tree and creates a TreeRetriever instance.

        Args:
            docs (str): The input text to add to the tree.
        """
        assert self.tree is None, "Tree already exists"

        self.tree = self.tree_builder.build_from_text(text=docs)
        self.retriever = TreeRetriever(
            self.tree_retriever_config,
            self.tree,
            retrieve_context_func=self.retrieve_context_func
        )

    def retrieve(
        self,
        question,
        start_layer: int = 1,
        num_layers: int = 2,
        top_k: int = 10,
        max_tokens: int = 3500,
        collapse_tree: bool = True,
        return_layer_information: bool = True,
    ):
        """
        Retrieves information and answers a question using the TreeRetriever instance.

        Args:
            question (str): The question to answer.
            start_layer (int): The layer to start from. Defaults to self.start_layer.
            num_layers (int): The number of layers to traverse. Defaults to self.num_layers.
            max_tokens (int): The maximum number of tokens. Defaults to 3500.
            use_all_information (bool): Whether to retrieve information from all nodes. Defaults to False.

        Returns:
            str: The context from which the answer can be found.

        Raises:
            ValueError: If the TreeRetriever instance has not been initialized.
        """
        if self.retriever is None:
            raise ValueError(
                "The TreeRetriever instance has not been initialized. Call 'add_documents' first."
            )

        return self.retriever.retrieve(
            question,
            start_layer,
            num_layers,
            top_k,
            max_tokens,
            collapse_tree,
            return_layer_information,
        )

    def answer_question(
        self,
        question,
        top_k: int = 1,
        start_layer: int = 1,
        num_layers: int = 2,
        max_tokens: int = 3500,
        collapse_tree: bool = False,
    ):
        """
        Retrieves information and answers a question using the TreeRetriever instance.

        Args:
            question (str): The question to answer.
            start_layer (int): The layer to start from. Defaults to self.start_layer.
            num_layers (int): The number of layers to traverse. Defaults to self.num_layers.
            max_tokens (int): The maximum number of tokens. Defaults to 3500.
            use_all_information (bool): Whether to retrieve information from all nodes. Defaults to False.
            ??

        Returns:
            str: The answer to the question.

        Raises:
            ValueError: If the TreeRetriever instance has not been initialized.
        """

        context = self.retrieve(
            question, start_layer, num_layers, top_k, max_tokens, collapse_tree, True
        )

        answer = self.qa_model.answer_question(context, question)

        return answer

    def save(self, path):
        if self.tree is None:
            raise ValueError("There is no tree to save.")
        with open(path, "wb") as file:
            pickle.dump(self.tree, file)
        logging.info(f"Tree successfully saved to {path}")

class TreeRetriever(BaseRetriever):

    def __init__(self, config, tree, retrieve_context_func=None) -> None:

        if config.num_layers is not None and config.num_layers > tree.num_layers + 1:
            raise ValueError(
                "num_layers in config must be less than or equal to tree.num_layers + 1"
            )

        if config.start_layer is not None and config.start_layer > tree.num_layers:
            raise ValueError(
                "start_layer in config must be less than or equal to tree.num_layers"
            )

        self.tree = tree
        self.num_layers = (
            config.num_layers
        )
        self.start_layer = (
            config.start_layer
        )

        self.tokenizer = config.tokenizer
        self.top_k = config.top_k
        self.threshold = config.threshold
        self.selection_mode = config.selection_mode
        self.embedding_model = config.embedding_model
        self.context_embedding_model = config.context_embedding_model

        if hasattr(self.tree, "layer_to_nodes"):
            self.tree_node_index_to_layer = reverse_mapping(self.tree.layer_to_nodes)

        logging.info(
            f"Successfully initialized TreeRetriever with Config {config.log_config()}"
        )
        self.retrieve_context_func = retrieve_context_func
        assert self.retrieve_context_func is not None, "retrieve_context_func must be provided"

    def create_embedding(self, text: str) -> List[float]:
        """
        Generates embeddings for the given text using the specified embedding model.

        Args:
            text (str): The text for which to generate embeddings.

        Returns:
            List[float]: The generated embeddings.
        """
        return self.embedding_model.create_embedding(text)

    def retrieve(
        self,
        query: str,
        start_layer: int = 2,
        num_layers: int = 1,
        top_k: int = 10,
        max_tokens: int = 3500,
        collapse_tree: bool = True,
        return_layer_information: bool = False,
    ) -> str:
        """
        Queries the tree and returns the most relevant information.

        Args:
            query (str): The query text.
            start_layer (int): The layer to start from. Defaults to self.start_layer.
            num_layers (int): The number of layers to traverse. Defaults to self.num_layers.
            max_tokens (int): The maximum number of tokens. Defaults to 3500.
            collapse_tree (bool): Whether to retrieve information from all nodes. Defaults to False.

        Returns:
            str: The result of the query.
        """

        if not isinstance(query, str):
            raise ValueError("query must be a string")

        if not isinstance(max_tokens, int) or max_tokens < 1:
            raise ValueError("max_tokens must be an integer and at least 1")

        if not isinstance(collapse_tree, bool):
            raise ValueError("collapse_tree must be a boolean")

        # Set defaults
        start_layer = self.start_layer if start_layer is None else start_layer
        num_layers = self.num_layers if num_layers is None else num_layers

        self.tree.num_layers = num_layers
        self.tree.start_layer = start_layer

        context = self.retrieve_context_func(query, self.tree, self.embedding_model)

        return context

# Setup

In [5]:
sample_text_path = os.path.join(SAMPLE_TEXT_PATH)
with open(sample_text_path, 'r') as file:
    text = file.read()

login(token=HF_TOKEN)

rac = RetrievalAugmentationConfig(
    summarization_model=GEMMASummarizationModel(),
    qa_model=GEMMAQAModel(),
    embedding_model=SBertEmbeddingModel(),
    tb_max_tokens=GRANULARITY,
    tb_summarization_length=SUMMARY_LENGTH
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/oh/arubinstein17/.cache/huggingface/token
Login successful


/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.11it/s]
2025-02-27 22:48:28,360 - Load pretrained SentenceTransformer: sentence-transformers/multi-qa-mpnet-base-cos-v1
2025-02-27 22:48:29,645 - Use pytorch device: cuda


# Standard RAG

### Flat-database builder

In [6]:
class Document:
    """
    Represents a document in the flat database.
    """

    def __init__(self, text: str, index: int, embedding) -> None:
        self.text = text
        self.index = index
        self.embedding = embedding


class FlatDatabase:
    """
    Represents the flat database structure.
    """

    def __init__(
        self,
        all_documents
    ) -> None:
        self.all_documents = all_documents


class FlatDatabaseBuilder:
    """
    ?? The TreeBuilder class is responsible for building a hierarchical text abstraction
    structure, known as a "tree," using summarization models and
    embedding models.
    """

    def __init__(self, embedding_model, tokenizer, max_tokens) -> None:
        """
        Initializes the tokenizer, maximum tokens, number of layers,
        top-k value, threshold, and selection mode.
        """

        self.tokenizer = tokenizer
        self.max_tokens = max_tokens
        self.embedding_model = embedding_model

    def create_document(self, index: int, text: str) -> Document:
        embedding = self.embedding_model(text)
        return Document(text, index, embedding)

    def build_from_text(self, text: str) -> FlatDatabase:
        """Builds a golden tree from the input text, optionally using multithreading.

        Args:
            text (str): The input text.

        Returns:
            Tree: The golden tree structure.
            ??
        """
        chunks = split_text(text, self.tokenizer, self.max_tokens)

        all_documents = [self.create_document(index, text) for index, text in enumerate(chunks)]
        flat_database = FlatDatabase(all_documents=all_documents)

        return flat_database


def make_flat_database_builder():
    return FlatDatabaseBuilder(
        embedding_model=SBertEmbeddingModel(),
        tokenizer=tiktoken.get_encoding("cl100k_base"),
        max_tokens=GRANULARITY
    )

## Retrieval from flat database

In [7]:
def retrieve_information_from_flat_database(
        query: str, database: Any, embedding_func: Callable, top_k: int = 1
    ) -> str:
        """
        Retrieves the most relevant information from the tree based on the query.

        Args:
            query (str): The query text.
            database (Any): The database with documents.
            embedding_func (Callable): The function to embed the documents.

        Returns:
            str: The context created using the most relevant documents.
        """

        documents = database.all_documents

        query_embedding = embedding_func(query)

        selected_nodes = []

        embeddings = [document.embedding for document in documents]

        distances = distances_from_embeddings(query_embedding, embeddings)

        indices = indices_of_nearest_neighbors_from_distances(distances)

        best_indices = indices[:top_k]

        nodes_to_add = [documents[idx] for idx in best_indices]

        selected_nodes.extend(nodes_to_add)

        return get_text(selected_nodes)

## Create flat database

In [8]:
ra_flat = RetrievalAugmentation(
    config=rac,
    tree=TREE_PATH,
    retrieve_context_func=retrieve_information_from_flat_database,
    make_database_builder_func=make_flat_database_builder
)

ra_flat.add_documents(text)

2025-02-27 22:48:33,470 - Load pretrained SentenceTransformer: sentence-transformers/multi-qa-mpnet-base-cos-v1
2025-02-27 22:48:34,571 - Use pytorch device: cuda
2025-02-27 22:48:34,572 - Successfully initialized RetrievalAugmentation with Config 
        RetrievalAugmentationConfig:
            
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 200
            Summarization Model: <__main__.GEMMASummarizationModel object at 0x7f6b224589a0>
            Embedding Models: {'EMB': <__main__.SBertEmbeddingModel object at 0x7f6b22429c90>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        

            
        TreeRetrieverConfig:
            Tokenizer: <Encoding 'cl100k_base'>


## Answer question

In [9]:
question = "What was the cause of Evelyn's symptoms?"

answer_flat = ra_flat.answer_question(
    question=question,
)

print("Answer with flat database: ", answer_flat)

Batches: 100%|██████████| 1/1 [00:00<00:00, 58.01it/s]
/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/oh/arubinstein17/github/raptor/envs/raptor/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Context:   She began exhibiting peculiar yet benign symptoms: an uncanny sense of direction, increased visual acuity, and, most curiously, a compulsive need to coo softly, particularly during the twilight hours  Neighbors joked kindly about Evelyn's newfound "birdsong," but soon others experienced similar changes Symptoms progressed gradually, typically beginning with heightened senses and remarkable navigation skills, but eventually escalating into more pronounced behavioral transformations


Question:  What was the cause of Evelyn's symptoms?
Answer with flat database:  The passage does not specify the cause of Evelyn's symptoms, so I cannot answer this question from the provided context.


# RAPTOR-based RAG

### [Task] Tree builder

In [10]:
class Node:
    """
    Represents a node in the hierarchical tree structure.
    """

    def __init__(self, text: str, index: int, children: Set[int], embeddings, summarization_prompt: Optional[str] = None) -> None:
        self.text = text
        self.index = index
        self.children = children
        self.embeddings = embeddings
        self.summarization_prompt = summarization_prompt

class Tree:
    """
    Represents the entire hierarchical tree structure.
    """

    def __init__(
        self, all_nodes, root_nodes, leaf_nodes, num_layers, layer_to_nodes, start_layer
    ) -> None:
        self.all_nodes = all_nodes
        self.root_nodes = root_nodes
        self.leaf_nodes = leaf_nodes
        self.num_layers = num_layers
        self.layer_to_nodes = layer_to_nodes
        self.start_layer = start_layer


class TreeBuilder:
    """
    The TreeBuilder class is responsible for building a hierarchical text abstraction
    structure, known as a "tree," using summarization models and
    embedding models.
    """

    def __init__(self, config) -> None:
        """Initializes the tokenizer, maximum tokens, number of layers, top-k value, threshold, and selection mode."""

        self.tokenizer = config.tokenizer
        self.max_tokens = config.max_tokens
        self.num_layers = config.num_layers
        self.start_layer = config.num_layers - 1
        self.top_k = config.top_k
        self.threshold = config.threshold
        self.selection_mode = config.selection_mode
        self.summarization_length = config.summarization_length
        self.summarization_model = config.summarization_model
        self.embedding_models = config.embedding_models
        self.cluster_embedding_model = config.cluster_embedding_model

        if not isinstance(config, ClusterTreeConfig):
            raise ValueError("config must be an instance of ClusterTreeConfig")
        self.reduction_dimension = config.reduction_dimension
        self.clustering_algorithm = config.clustering_algorithm
        self.clustering_params = config.clustering_params

        logging.info(
            f"Successfully initialized TreeBuilder with Config {config.log_config()}"
        )

    def create_node(
        self, index: int, text: str, children_indices: Optional[Set[int]] = None, summarization_prompt: Optional[str] = None
    ) -> Tuple[int, Node]:
        """Creates a new node with the given index, text, and (optionally) children indices.

        Args:
            index (int): The index of the new node.
            text (str): The text associated with the new node.
            children_indices (Optional[Set[int]]): A set of indices representing the children of the new node.
                If not provided, an empty set will be used.

        Returns:
            Tuple[int, Node]: A tuple containing the index and the newly created node.
        """
        if children_indices is None:
            children_indices = set()

        # Do we need model names dict?
        embeddings = {
            model_name: model.create_embedding(text)
            for model_name, model in self.embedding_models.items()
        }
        return (index, Node(text, index, children_indices, embeddings, summarization_prompt))

    def create_embedding(self, text) -> List[float]:
        """
        Generates embeddings for the given text using the specified embedding model.

        Args:
            text (str): The text for which to generate embeddings.

        Returns:
            List[float]: The generated embeddings.
        """
        return self.embedding_models[self.cluster_embedding_model].create_embedding(
            text
        )

    def summarize(self, context, max_tokens=150) -> str:
        """
        Generates a summary of the input context using the specified summarization model.

        Args:
            context (str, optional): The context to summarize.
            max_tokens (int, optional): The maximum number of tokens in the generated summary. Defaults to 150.o

        Returns:
            str: The generated summary.
        """
        return self.summarization_model.summarize(context, max_tokens)

    def build_from_text(self, text: str) -> Tree:
        """Builds a golden tree from the input text, optionally using multithreading.

        Args:
            text (str): The input text.

        Returns:
            Tree: The golden tree structure.
        """
        chunks = split_text(text, self.tokenizer, self.max_tokens)

        logging.info("Creating Leaf Nodes")

        leaf_nodes = {}

        # INSERT YOUR CODE BELOW (use self.create_node to create leaf nodes)
        for index, text in enumerate(chunks):
            __, node = self.create_node(index, text)
            leaf_nodes[index] = node

        # INSERT YOUR CODE ABOVE

        layer_to_nodes = {0: list(leaf_nodes.values())}

        logging.info(f"Created {len(leaf_nodes)} Leaf Embeddings")

        logging.info("Building All Nodes")

        all_nodes = copy.deepcopy(leaf_nodes)

        # INSERT YOUR CODE BELOW (use self.construct_tree to create non-leaf nodes)

        root_nodes = self.construct_tree(all_nodes, all_nodes, layer_to_nodes)

        # INSERT YOUR CODE ABOVE

        tree = Tree(all_nodes, root_nodes, leaf_nodes, self.num_layers, layer_to_nodes, self.start_layer)

        return tree

    def construct_tree(
        self,
        current_level_nodes: Dict[int, Node],
        all_tree_nodes: Dict[int, Node],
        layer_to_nodes: Dict[int, List[Node]],
    ) -> Dict[int, Node]:
        logging.info("Using Cluster TreeBuilder")

        next_node_index = len(all_tree_nodes)

        def process_cluster(
            cluster,
            new_level_nodes,
            next_node_index,
            summarization_length,
        ):
            node_texts = get_text(cluster)

            # INSERT YOUR CODE BELOW (use self.summarize)

            summarized_text = self.summarize(
                context=node_texts,
                max_tokens=summarization_length,
            )

            # INSERT YOUR CODE ABOVE

            if isinstance(summarized_text, tuple):
                assert len(summarized_text) == 2
                summarized_text, summarization_prompt = summarized_text
            else:
                summarization_prompt = None


            # INSERT YOUR CODE BELOW (use self.create_node)

            logging.info(
                f"Node Texts Length: {len(self.tokenizer.encode(node_texts))}, Summarized Text Length: {len(self.tokenizer.encode(summarized_text))}"
            )

            __, new_parent_node = self.create_node(
                index=next_node_index,
                text=summarized_text,
                children_indices={node.index for node in cluster},
                summarization_prompt=summarization_prompt
            )

            # INSERT YOUR CODE ABOVE

            new_level_nodes[next_node_index] = new_parent_node

        for layer in range(self.num_layers):

            new_level_nodes = {}

            logging.info(f"Constructing Layer {layer}")

            node_list_current_layer = get_node_list(current_level_nodes)

            if len(node_list_current_layer) <= self.reduction_dimension + 1:
                self.num_layers = layer
                logging.info(
                    f"Stopping Layer construction: Cannot Create More Layers. Total Layers in tree: {layer}"
                )
                break

            clusters = self.clustering_algorithm.perform_clustering(
                node_list_current_layer,
                self.cluster_embedding_model,
                reduction_dimension=self.reduction_dimension,
                **self.clustering_params,
            )

            summarization_length = self.summarization_length
            logging.info(f"Summarization Length: {summarization_length}")

            # INSERT YOUR CODE BELOW (use process_cluster)

            for cluster in clusters:
                process_cluster(
                    cluster,
                    new_level_nodes,
                    next_node_index,
                    summarization_length,
                )
                next_node_index += 1

            # INSERT YOUR CODE ABOVE

            layer_to_nodes[layer + 1] = list(new_level_nodes.values())
            current_level_nodes = new_level_nodes
            all_tree_nodes.update(new_level_nodes)

            tree = Tree(
                all_tree_nodes,
                layer_to_nodes[layer + 1],
                layer_to_nodes[0],
                layer + 1,
                layer_to_nodes,
                start_layer=self.start_layer
            )

        return current_level_nodes


def prepare_tree_builder_maker(ra_config):

    def make_tree_builder():
        return TreeBuilder(ra_config.tree_builder_config)

    return make_tree_builder

### [Task] Retrieval from tree

In [11]:
def retrieve_information_from_tree(
        query: str, database: Any, embedding_func: Callable, top_k: int = 1
    ) -> str:
        """
        Retrieves the most relevant information from the tree based on the query.

        Args:
            query (str): The query text.
            database (Any): The database with documents.
            embedding_func (Callable): The function to embed the documents.

        Returns:
            str: The context created using the most relevant documents.
        """
        start_layer = database.start_layer
        num_layers = database.num_layers

        if num_layers > start_layer + 1:
            raise ValueError("num_layers must be less than or equal to start_layer + 1")

        if not isinstance(start_layer, int) or not (
            0 <= start_layer <= num_layers
        ):
            raise ValueError(
                "start_layer must be an integer between 0 and tree.num_layers"
            )

        if not isinstance(num_layers, int) or num_layers < 1:
            raise ValueError("num_layers must be an integer and at least 1")

        if num_layers > (start_layer + 1):
            raise ValueError("num_layers must be less than or equal to start_layer + 1")

        # INSERT YOUR CODE BELOW
        print("SOLUTION FOR retrieve_information")

        node_list = database.layer_to_nodes[start_layer]

        query_embedding = embedding_func(query)

        selected_nodes = []

        for layer in range(num_layers):

            embeddings = get_embeddings(
                node_list,
                embedding_model_name="EMB"
            )

            distances = distances_from_embeddings(query_embedding, embeddings)

            indices = indices_of_nearest_neighbors_from_distances(distances)

            best_indices = indices[: top_k]

            nodes_to_add = [node_list[idx] for idx in best_indices]

            selected_nodes.extend(nodes_to_add)

            if layer != num_layers - 1:

                child_nodes = []

                for index in best_indices:
                    child_nodes.extend(node_list[index].children)

                # take the unique values
                child_nodes = list(dict.fromkeys(child_nodes))
                node_list = [database.all_nodes[i] for i in child_nodes]

        context = get_text(selected_nodes)
        # INSERT YOUR CODE ABOVE

        return context


def get_embeddings(node_list: List[Node], embedding_model_name: str) -> List:
    """
    Extracts the embeddings of nodes from a list of nodes.

    Args:
        node_list (List[Node]): List of nodes to extract embeddings from.
        embedding_model_name (str): Name of embedding model to use.

    Returns:
        List: List of node embeddings.
    """
    return [node.embeddings[embedding_model_name] for node in node_list]

### Create tree

In [12]:
ra_tree = RetrievalAugmentation(
    config=rac,
    tree=TREE_PATH,
    retrieve_context_func=retrieve_information_from_tree,
    make_database_builder_func=prepare_tree_builder_maker(rac)
)

ra_tree.add_documents(text)
# cell execution time: ~1min

2025-02-27 22:48:47,413 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 200
            Summarization Model: <__main__.GEMMASummarizationModel object at 0x7f6b224589a0>
            Embedding Models: {'EMB': <__main__.SBertEmbeddingModel object at 0x7f6b22429c90>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2025-02-27 22:48:47,415 - Successfully initialized RetrievalAugmentation with Config 
        RetrievalAugmentationConfig:
            
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
 

### Optionally save tree

In [13]:
to_save_tree = False # set to True to save tree

if to_save_tree:

    assert text is not None, "text must be provided"
    assert ra_tree is not None, "ra_tree must be provided"

    if not os.path.exists(TREE_PATH):

        ra_tree.save(TREE_PATH)

### Answer question

In [14]:
question = "What was the cause of Evelyn's symptoms?"

answer_tree = ra_tree.answer_question(
    question=question,
)

print("Answer with tree database: ", answer_tree)

SOLUTION FOR retrieve_information


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.30it/s]

Context:  Sure, here's a summary of the passage:  Evelyn, a seemingly healthy individual, began exhibiting peculiar symptoms that gradually progressed from heightened senses to more pronounced behavioral transformations. Her symptoms included an uncanny sense of direction, increased visual acuity, and a compulsive need to coo softly, particularly during twilight hours.  The symptoms were initially observed by neighbors, but they were initially dismissed as harmless. However, as the symptoms persisted, they became more pronounced, leading to identity struggles and social isolation.  A collaboration between Dr. Clara Novak and Dr. Liam Greer revealed that the cause of Evelyn's symptoms was a fungus called Columba benedicta, which released spores that affected the human nervous system.  Treatment involved antifungal medications, cognitive-behavioral therapy, nasal spray, and topical cream. Additional treatments included intravenous antifungal therapy for severe cases.  The condition was m

Answer with tree database:  The cause of Evelyn's symptoms was a fungus called Columba benedicta, which released spores that affected the human nervous system.
